# Agglomerative hierarchical clustering
The proximity matrix by hand, the four linkages, and the shopping data: dendrogram, cut, 5 clusters.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from sklearn.cluster import AgglomerativeClustering

## The proximity matrix

In [ ]:
P = np.array([[1, 1], [2, 2], [5, 4], [6, 4], [6, 6]])
D = cdist(P, P)                       # distance between every pair of points: a 5 x 5 matrix
pd.DataFrame(D.round(2), index=["P1", "P2", "P3", "P4", "P5"], columns=["P1", "P2", "P3", "P4", "P5"])

In [ ]:
# The merges, one row each: [cluster a, cluster b, distance, size]; ids 5, 6, ... are new clusters
for method in ["single", "complete", "average", "ward"]:
    print(method, "\n", linkage(P, method=method).round(2))

## The shopping data

In [ ]:
customers = pd.read_csv("data/shopping_data.csv")
print(customers.shape)
customers.head()

In [ ]:
# Keep two columns: annual income and spending score
data = customers.iloc[:, 3:5].values

In [ ]:
# The dendrogram: scipy computes it, Plotly draws it
Z = linkage(data, method="ward")
d = dendrogram(Z, no_plot=True)      # coordinates only, no plotting library needed
fig = go.Figure()
for xs, ys in zip(d["icoord"], d["dcoord"]):
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#4C78A8", width=1), showlegend=False))
fig.add_hline(y=180, line_dash="dash", line_color="red")
fig.update_layout(template="simple_white", title="Customer dendrogram", xaxis_showticklabels=False,
                  yaxis_title="merge distance")

In [ ]:
# The last merge heights, largest first: the gaps between them are the uncut vertical lines
print(np.sort(Z[:, 2])[::-1][:6].round(1))

In [ ]:
cluster = AgglomerativeClustering(n_clusters=5, metric="euclidean", linkage="ward")
labels_ = cluster.fit_predict(data)
print(labels_[:20])
print("customers per cluster:", np.bincount(labels_))

In [ ]:
fig = go.Figure()
for k in range(5):
    fig.add_trace(go.Scatter(x=data[labels_ == k, 0], y=data[labels_ == k, 1], mode="markers", name=f"cluster {k}"))
fig.update_layout(template="simple_white", xaxis_title="annual income (k$)", yaxis_title="spending score")

In [ ]:
# Instead of n_clusters, a distance threshold: stop merging above this height
cut = AgglomerativeClustering(n_clusters=None, distance_threshold=180, linkage="ward").fit(data)
print("clusters below 180:", cut.n_clusters_)